# Kata 2 — Deterministic Guardrails via PreToolUse Hooks

**Claude Architecture Certification — reference notebook**

This notebook is the Principle VIII deliverable for Kata 2. It folds objective, walkthrough, anti-pattern defense, run instructions, and reflection into one place — there is no separate `README.md` for this kata.

> **TL;DR.** Move the refund threshold OUT of the system prompt and INTO a deterministic `PreToolUseHook` that runs *before* the SDK dispatches `process_refund`. Validate the payload with pydantic, compare amount against `PolicyConfig.max_refund` from `config/policy.json`, and return a typed `StructuredError` on reject — never free text. Zero external API calls escape on a reject. The policy limit is data, not code, and the absence of any numeric literal in `prompts.py` is enforced by an AST lint.

## Section 1 — Objective & anti-pattern

**Objective.** Make the refund policy *physically unbypassable*. A schema-valid in-policy refund must reach the external refund API and return a real success outcome; a refund that exceeds the policy limit must be blocked at the PreToolUse boundary with a structured error and a human-routed escalation event, with zero external API calls. Editing `config/policy.json` from L1 to L2 must take effect on the next invocation, with no code, prompt, or schema change.

**Anti-pattern this kata structurally defends against.** *Prompt-only enforcement* — phrasing the rule as instructions inside the system prompt ("do not refund more than $500"). This fails because (a) the model paraphrases the rule under load, (b) attention dilution drops the rule mid-context after enough turns, and (c) the rule lives in free text rather than executable code. Kata 2 makes the rule structural: even a model that decides to ignore the prompt cannot make the API call, because the call is gated by a `PreToolUseHook` that runs Python, not English.

## Section 2 — Concepts (Claude architecture certification)

Every concept this kata exercises, with a one-line definition tied to the certification syllabus and the file that demonstrates it.

| Concept | One-line definition | Where it lives in this kata |
|---|---|---|
| **PreToolUse hook** | A deterministic gate that runs *before* the SDK dispatches a tool, with the authority to block the call and substitute a structured error. | `hook.py::RefundPolicyHook.evaluate` |
| **Deterministic guardrails** | Allow/reject decisions are functions of `(payload, policy_snapshot)` only — same input ⇒ same output. | `models.py::HookVerdict` + `unit/test_determinism_repeat_run.py` |
| **Structured tool-result errors** | The model sees a typed `StructuredError`, never free-text apology, on reject. | `models.py::StructuredError` + `runner.build_structured_error` |
| **Policy-as-data + hot reload** | The threshold lives in `config/policy.json`; the loader reads fresh on every call so a file edit takes effect on the next invocation. | `policy.py::load_policy` + `unit/test_policy_change_takes_effect.py` |
| **Fail-closed control flow** | A hook that cannot evaluate (corrupt policy file, unexpected exception) rejects with `reason_code="hook_failure"` — never silently allows. | `hook.py` `try/except` + `unit/test_hook_failure_failsafe.py` |
| **Audit log / JSONL provenance** | One line per invocation, verdict, escalation, and refund-API call lands under `runs/<session>/`; the absence of a `refund_api_call` line on reject is the SC-002 zero-call assertion. | `events.py::EventLog` + `refund_api_stub.py` |
| **Human-in-the-loop escalation** | Every `policy_breach` and `hook_failure` reject emits an `EscalationEvent` with `actions_taken=[]` (no action was taken — by construction). | `escalation.py::build_escalation_event` |
| **AST lint as machine-enforced policy** | A lint test parses `prompts.py` and fails if any string constant contains a numeric literal equal to the active `max_refund`. | `tests/.../lint/test_prompt_has_no_limit.py` |
| **Decimal-only money path** | The amount field uses `decimal.Decimal` end-to-end; `float` is rejected at construction time and lint-blocked from the source. | `models.py::ToolCallPayload._reject_float_amount` + `tests/.../lint/test_no_float_in_amount_path.py` |
| **Snapshot-at-entry under concurrent edits** | The verdict pins the `policy_snapshot_version` it evaluated against, even if `config/policy.json` is rewritten before the audit log lands. | `unit/test_concurrent_policy_snapshot.py` |

## Section 3 — Architecture walkthrough

```mermaid
flowchart LR
    Agent[Agent runtime / runner.py] --> Hook[RefundPolicyHook]
    Hook --> Pol[(config/policy.json<br/>PolicyConfig)]
    Hook -- allow --> Stub[refund_api_stub.process_refund]
    Hook -- reject --> SE[StructuredError -> model context]
    Hook -- reject* --> Esc[EscalationEvent]
    Stub --> RC[(refund_api_calls.jsonl)]
    Hook --> EL[(events.jsonl<br/>EventLog)]
    Esc --> EL
```

ASCII fallback:
```
+-----------------------+
|     runner.py         |
+----------+------------+
           |
+----------v------------+      +-----------------+
|  RefundPolicyHook     | ---> |  PolicyConfig   |
+----------+------------+      |  (per-invocation
           |                   |   reload)       |
  +--------+-------+           +-----------------+
  | allow          |
  v                v
 refund_api_stub  StructuredError -> model context
  |                +-> EscalationEvent (policy_breach / hook_failure only)
  v
 refund_api_calls.jsonl    events.jsonl  (every invocation, verdict, escalation, refund_api_call)
```

### Module map

| Module | Role | Why this seam exists |
|---|---|---|
| `runner.py` | CLI + end-to-end orchestrator. Handles policy load, hook eval, stub dispatch, audit log, escalation. | Concentrates wiring so each individual module stays single-purpose. |
| `hook.py` | `PreToolUseHook` protocol + `RefundPolicyHook` concrete. Three-step decision pipeline. | Single decision point makes Principle I auditable. |
| `models.py` | Pydantic v2 boundary models (`ToolCallPayload`, `PolicyConfig`, `HookVerdict`, `StructuredError`, `EscalationEvent`). | Schema-first contract — Principle II. |
| `errors.py` | `ReasonCode` enum + the narrower `ESCALATING_REASON_CODES` subset. | Explicit enums; no string magic. |
| `policy.py` | `load_policy` + `PolicyLoadError`. Per-invocation reload. | FR-011 / SC-004 hot reload without caching. |
| `escalation.py` | `build_escalation_event` + JSONL emit. | Principle VI escalation, narrowed to the two causes that escalate. |
| `events.py` | Append-only JSONL `EventLog`. | Principle VII provenance — log alone reconstructs the run. |
| `refund_api_stub.py` | Local stand-in for the external refund API. Records every call to disk. | Filesystem evidence for SC-002 zero-call assertions. |
| `prompts.py` | Role + tool-use guidance only — *no numeric literals*. | The AST lint here is the structural FR-008 guarantee. |

## Section 4 — Patterns and the trade-off each one solves

| Pattern | What it does | Trade-off it solves |
|---|---|---|
| **Guardrail-before-tool** | Run a `PreToolUseHook` synchronously before the SDK dispatches the tool call. | Prompt-only enforcement fails under paraphrase; a hook gate is structural. |
| **Schema-first contract** | Every boundary object is pydantic v2 with `extra="forbid"` + a JSON-Schema mirror. | Drift between modules caught at construction; absent schemas push errors to production. |
| **Fail-closed reject** | Any uncaught exception in the hook becomes `HookVerdict(reject, hook_failure)`. | Silent fail-open re-introduces exactly the gap the kata closes. |
| **Snapshot-at-entry for policy** | Capture `PolicyConfig` once at the top of `evaluate`; verdict and audit log both cite that version. | Eliminates the concurrent-policy-update race; reviewer reads one snapshot per correlation_id. |
| **Structured-error hand-back to model context** | Reject path returns a typed `StructuredError`, not prose. | Free text lets the model paraphrase the rejection and retry; structure forces a deterministic next turn. |
| **Decimal-only money path** | `Decimal` end-to-end; `float()` lint-banned in `hook.py / models.py / runner.py`. | `0.1 + 0.2 != 0.3` would non-determinise the policy comparison. |
| **AST lint as structural guard** | Walk `prompts.py` AST and fail on any literal equal to the current `max_refund`. | Reviewer-enforced rules drift; AST checks fail the build before merge. |

## Section 5 — Principles enforced + Anthropic engineering recommendations

### Constitution → file-level enforcement

| Principle | What it requires | Where it is enforced |
|---|---|---|
| **II. Schema-Enforced Boundaries** *(NN)* | Every boundary object is a typed schema. | `models.py` + `specs/002-pretool-guardrails/contracts/*.schema.json` + `step_defs/test_contracts_payloads_steps.py` |
| **VI. Human-in-the-Loop Escalation** *(NN)* | Reject paths emit a typed escalation payload. | `escalation.py::build_escalation_event` + `unit/test_escalation_event_emitted.py` |
| **VII. Provenance & Self-Audit** | Append-only audit log reconstructible offline. | `events.py::EventLog` |
| **VIII. Mandatory Documentation** *(NN)* | Why-comments on non-trivial code; a written deliverable per kata. | Module docstrings, why-comments throughout, this notebook. |

### Anthropic engineering recommendations exercised

- **Use deterministic guardrails outside the prompt to enforce business policy.** This kata is the canonical demonstration.
- **Validate tool inputs with declared JSON schemas before dispatch.** `ToolCallPayload` is checked first; only after that does the policy comparison run.
- **Return structured errors into the model's next context window.** `StructuredError` is the only rejection shape the model sees.
- **Make agent runs auditable.** One JSONL line per invocation, verdict, escalation, and refund-API call.

### Practitioner checklist — applying PreToolUse guardrails on a real refund / payments path

1. **List every reject reason** *before* writing the hook. If you don't have a labelled bucket for `hook_failure`, you'll silently allow operational outages.
2. **Move every business threshold into a versioned config file.** Reload it per invocation; never cache across calls.
3. **Make the audit log the ground truth.** One JSONL line per kind; absence of a record is itself an assertion.
4. **Forbid the threshold inside any prompt module by lint.** AST checks are cheap and catch the regression before review.
5. **Use `Decimal` for money end-to-end.** Lint `float(...)` calls and `: float` annotations on the amount path.
6. **Pin the policy snapshot at entry.** Both the verdict and the escalation cite the same version, regardless of concurrent edits.
7. **Hand the model a typed rejection.** Free text lets the model paraphrase; structure forces deterministic next-turn behaviour.
8. **Escalate on `policy_breach` and `hook_failure`, not on `schema_violation`.** Schema bugs are client-side; route them to `client_fix_required`.

## Section 6 — Hook contract

**Input.** `ToolCallPayload` ([schema](../../specs/002-pretool-guardrails/contracts/tool-call-payload.schema.json)) — `tool_name='process_refund'`, `correlation_id` (UUID), `amount` (Decimal-safe string), `currency='USD'`, `customer_id`, optional `reason`. `extra="forbid"` rejects any unknown field as `schema_violation`.

**Output.** `HookVerdict` ([schema](../../specs/002-pretool-guardrails/contracts/hook-verdict.schema.json)) — `verdict ∈ {"allow", "reject"}`. On reject, `reason_code` is one of:

| Reason code | Meaning | Escalates? | Pathway |
|---|---|---|---|
| `schema_violation` | Payload failed pydantic validation (client bug). | No (D-007). | `client_fix_required` |
| `policy_breach` | Payload was well-formed; `amount >= max_refund` under `strict_less_than`. | Yes. | `PolicyConfig.escalation_pathway` |
| `hook_failure` | The hook itself raised (corrupt policy, unexpected exception). | Yes. | `hook-failure-oncall` |

**Runner CLI exit codes.**

| Exit | Meaning |
|---|---|
| `0` | allow + stub success |
| `10` | reject — `schema_violation` |
| `11` | reject — `policy_breach` |
| `20` | reject — `hook_failure` |

**Worked example — over-limit reject.** Input `{amount: "750.00", customer_id: "C-1003", ...}` against `max_refund="500.00"` ⇒ `HookVerdict(reject, policy_breach, offending_field="amount", offending_value="750.00")`; `StructuredError(reason_code="policy_breach", rule_violated="less_than_max_refund", policy_id="refund-policy", policy_snapshot_version="v1", escalation_pathway="refund-review-queue", message="Refund rejected by policy refund-policy v1: amount 750.00 exceeds the configured limit. Route the customer to the escalation pathway.")`; `events.jsonl` carries one `invocation`, one `verdict`, one `escalation`; `refund_api_calls.jsonl` is empty.

## Section 7 — Policy schema

`PolicyConfig` ([schema](../../specs/002-pretool-guardrails/contracts/policy-config.schema.json)) is loaded fresh from `config/policy.json` on every invocation.

| Field | Type | Invariant |
|---|---|---|
| `policy_id` | `str` | Stable identifier (e.g. `refund-policy`). |
| `policy_snapshot_version` | `str` | Monotonic version string; changes on every edit; recorded on every verdict / escalation. |
| `max_refund` | `Decimal` | The ONLY runtime source of the threshold; never duplicated into any prompt. |
| `comparison_stance` | `Literal["strict_less_than"]` | Declared stance: `amount < max_refund` passes; `amount == max_refund` rejects. |
| `escalation_pathway` | `str` | Routing target written into `StructuredError` and `EscalationEvent`. |
| `effective_from` | `datetime` (UTC) | When this snapshot became active; recorded on the audit log. |

Cross-link: every field above maps 1:1 to `specs/002-pretool-guardrails/contracts/policy-config.schema.json`.

## Section 8 — Escalation flow

When a reject verdict has `reason_code ∈ {policy_breach, hook_failure}`:

1. The runner builds an `EscalationEvent` from `(verdict, policy, payload, payload_digest)`.
2. `actions_taken` is fixed to `[]` — by construction, no API call has been made.
3. `rejected_payload_digest` is the SHA-256 of the JSON-serialised `ToolCallPayload`; the raw payload is never written to disk to avoid leaking PII into the audit log.
4. The event is appended to `runs/<session>/events.jsonl` with `kind="escalation"`.
5. A reviewer reading the JSONL groups records by `correlation_id` to recover `(invocation, verdict, escalation)` for the rejected refund and routes the customer to `routing_target` (e.g. `refund-review-queue`).

`schema_violation` rejects do NOT escalate — they go back to the model with `escalation_pathway="client_fix_required"`, telling the agent to re-issue a valid payload.

In [ ]:
# Section 9 — Run
# Boilerplate: make the kata package importable from this notebook regardless
# of where Jupyter is launched from.
from __future__ import annotations

import json
import sys
import uuid
from pathlib import Path

here = Path.cwd()
while not (here / "pyproject.toml").exists() and here != here.parent:
    here = here.parent
REPO_ROOT = here
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FIXTURES = REPO_ROOT / "tests" / "katas" / "kata_002_pretool_guardrails" / "fixtures"
POLICY = REPO_ROOT / "config" / "policy.json"
RUNS_ROOT = REPO_ROOT / "runs" / "notebook_kata_002"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

from katas.kata_002_pretool_guardrails.runner import run_once

print("repo root:", REPO_ROOT)
print("policy file:", POLICY)

In [ ]:
# Allow path: within_limit.json -> exactly one stub call, real success outcome.
session_within = RUNS_ROOT / f"within-{uuid.uuid4()}"
verdict, structured, success = run_once(
    payload_dict=json.loads((FIXTURES / "within_limit.json").read_text()),
    policy_path=POLICY,
    session_dir=session_within,
)
print("verdict:", verdict.verdict, "reason_code:", verdict.reason_code)
print("structured_error:", structured)
print("success:", success)
calls = (session_within / "refund_api_calls.jsonl").read_text().splitlines()
print(f"refund_api_calls.jsonl lines: {len(calls)}")
assert verdict.verdict == "allow"
assert len(calls) == 1
assert success and success["source"] == "refund_api_stub"

In [ ]:
# Reject path: over_limit.json -> zero stub calls, structured error, escalation.
session_over = RUNS_ROOT / f"over-{uuid.uuid4()}"
verdict, structured, success = run_once(
    payload_dict=json.loads((FIXTURES / "over_limit.json").read_text()),
    policy_path=POLICY,
    session_dir=session_over,
)
print("verdict:", verdict.verdict, "reason_code:", verdict.reason_code)
print("structured_error.reason_code:", structured.reason_code if structured else None)
calls_path = session_over / "refund_api_calls.jsonl"
assert verdict.verdict == "reject"
assert verdict.reason_code.value == "policy_breach"
assert success is None
assert not calls_path.exists() or calls_path.read_text().strip() == ""
events = [
    json.loads(line)
    for line in (session_over / "events.jsonl").read_text().splitlines()
    if line.strip()
]
kinds = [e["kind"] for e in events]
print("events.jsonl kinds:", kinds)
assert kinds.count("invocation") == 1
assert kinds.count("verdict") == 1
assert kinds.count("escalation") == 1
assert kinds.count("refund_api_call") == 0  # SC-002 zero-call assertion.

In [ ]:
# LIVE_API path (commented — opt-in, requires ANTHROPIC_API_KEY).
#
# When LIVE_API=1 is set, a future variant of runner.py will:
#   1. Send the system prompt + tool definitions to claude-opus-4-7.
#   2. Receive a tool_use block proposing process_refund(amount=750.00, ...).
#   3. Run the same RefundPolicyHook *before* the SDK dispatches the call.
#   4. Hand the StructuredError back to the model in the next tool_result block.
#   5. Observe that the model's next turn summarizes the rejection and routes
#      the customer to the escalation_pathway, instead of paraphrasing and
#      re-trying with a smaller amount.
#
# This kata's MVP is fixture-only — the LIVE path is exercised by Kata 4
# (subagent isolation), which builds on this hook.
print("LIVE_API path is opt-in; see kata 4 for the end-to-end Anthropic SDK loop.")

## Section 10 — Result

Captured outputs from the cells above:

**Allow path** (`within_limit.json` against `max_refund=500.00`)
- `verdict.verdict == "allow"`
- `success.source == "refund_api_stub"`
- `refund_api_calls.jsonl` has exactly one line
- `events.jsonl` has `invocation`, `verdict`, and `refund_api_call` records — in that order, joined by `correlation_id`

**Reject path** (`over_limit.json` against `max_refund=500.00`)
- `verdict.verdict == "reject"`, `reason_code == "policy_breach"`
- `structured_error.reason_code == "policy_breach"`, `rule_violated == "less_than_max_refund"`
- `refund_api_calls.jsonl` is **empty** — SC-002 holds
- `events.jsonl` has `invocation`, `verdict`, `escalation` (no `refund_api_call`)

**Test suite** (run separately):

```
45 tests pass — 16 unit tests, 2 AST lint tests, 5 BDD step modules
covering 23 test-spec IDs (TS-001 … TS-023) and the 13 functional
requirements FR-001 … FR-013.
```

## Section 11 — Reflection (Principle VIII)

Answers to the prompts in `quickstart.md`.

**Q: Where in the source tree is the number 500 allowed to appear, and where is it forbidden? How does the AST lint police that boundary?**

Allowed locations: `config/policy.json` (the runtime source of truth), `tests/katas/kata_002_pretool_guardrails/fixtures/*.json` (recorded payloads — these are *data*, not policy), `tests/katas/kata_002_pretool_guardrails/unit/*.py` (unit tests build payloads with explicit Decimals), and the docstring of `prompts.py` (which deliberately explains the rule by negative example). Forbidden location: any string literal *inside* `prompts.py` that is not the module docstring. The AST lint walks `prompts.py` with `ast.walk`, identifies every `ast.Constant` whose value is a string, excludes the module docstring constant, and runs a regex over the survivors looking for tokens equivalent to the current `max_refund` (500, 500.00, $500, USD 500). The lint reads the *current* `config/policy.json` so that lowering the policy to 250 immediately tightens the lint without any code change. (Principle I + Principle VIII.)

**Q: When the hook fails internally (corrupt policy file), why does the system fail closed rather than open, and which principle dictates the choice?**

Constitution Principle VI (Human-in-the-Loop Escalation, NN). A silent fail-open on hook errors would re-introduce exactly the gap the kata closes — an agent runtime that says "the hook didn't object" when the hook never had a chance to evaluate. Fail-closed translates *operational* failure into a typed `EscalationEvent(escalation_reason="hook_failure")` with a distinct routing target (`hook-failure-oncall`), so on-call sees "the hook itself broke" instead of "policy was breached" and the customer's refund is held until the hook is repaired. This is asserted by `unit/test_hook_failure_failsafe.py` — corrupt the snapshot mid-test, and the verdict must reject with `reason_code="hook_failure"` and zero stub calls.

**Q: If Compliance needs to tighten the limit to $250 tomorrow at 09:00, which exact file do they edit and what happens at 09:01 with zero redeploys?**

They edit `config/policy.json` — and only that file. Specifically: bump `policy_snapshot_version` (e.g. `"v1"` → `"v2-2026-04-28"`), set `max_refund` to `"250.00"`, refresh `effective_from`. At 09:01 the next refund invocation calls `load_policy()` which reads the file fresh; any amount in `[250, 500)` that was previously allowed now rejects with `reason_code="policy_breach"` and the new snapshot version on the verdict, structured error, and escalation event. No source file is touched: `prompts.py` keeps its byte-digest (assertion in `unit/test_policy_change_no_prompt_or_schema_edit.py`), the contract schemas are unchanged, the runner is not restarted. (Principle VIII reflection.)

**Q: What was the observed failure mode this kata was designed to prevent?**

The teaching anecdote: *"the model agreed not to refund >$500 five times; the sixth time it paraphrased the policy and refunded anyway."* That is the prompt-only-enforcement anti-pattern — the rule lived in prose, attention dilution dropped it after enough turns, and the agent confidently violated it. The hook now makes that physically impossible: even a model that decides to ignore the prompt cannot execute the refund, because the refund is gated by deterministic Python that runs *before* the SDK dispatches the call.